In [2]:
import sys
sys.path.append("/n/home12/binxuwang/Github/DiffusionAttnConsistency")
import os 
import torch as th
import numpy as np
from core.parity_lib import sample_group_parity_vec, sample_ensuring_uniqueness

In [3]:
# import numpy as np
# from core.parity_lib import sample_group_parity_vec
sample_num = 4096
sample_len = 64
for group_size in [2, 4, 8, 16, 32, 64]:
    x = sample_ensuring_uniqueness(N=sample_num, sample_func=lambda N: sample_group_parity_vec(N=N, sample_len=sample_len, group_size=group_size, parity=0))
    num_unique = len(np.unique(x, axis=0))
    assert num_unique == sample_num
    print(f"Even parity, group_size: {group_size}, num_unique: {num_unique}")
    x = sample_ensuring_uniqueness(N=sample_num, sample_func=lambda N: sample_group_parity_vec(N=N, sample_len=sample_len, group_size=group_size, parity=1))
    num_unique = len(np.unique(x, axis=0))
    assert num_unique == sample_num
    print(f"Odd parity,  group_size: {group_size}, num_unique: {num_unique}")

Even parity, group_size: 2, num_unique: 4096
Odd parity,  group_size: 2, num_unique: 4096
Even parity, group_size: 4, num_unique: 4096
Odd parity,  group_size: 4, num_unique: 4096
Even parity, group_size: 8, num_unique: 4096
Odd parity,  group_size: 8, num_unique: 4096
Even parity, group_size: 16, num_unique: 4096
Odd parity,  group_size: 16, num_unique: 4096
Even parity, group_size: 32, num_unique: 4096
Odd parity,  group_size: 32, num_unique: 4096
Even parity, group_size: 64, num_unique: 4096
Odd parity,  group_size: 64, num_unique: 4096


In [4]:
# import numpy as np
# from core.parity_lib import sample_group_parity_vec

sample_len = 36
for group_size in [2, 3, 4, 6, 9, 12, 18, 36]:
    x = sample_ensuring_uniqueness(N=4096, sample_func=lambda N: sample_group_parity_vec(N=N, sample_len=sample_len, group_size=group_size, parity=0))
    num_unique = len(np.unique(x, axis=0))
    print(f"Even parity, group_size: {group_size}, num_unique: {num_unique}")
    x = sample_ensuring_uniqueness(N=4096, sample_func=lambda N: sample_group_parity_vec(N=N, sample_len=sample_len, group_size=group_size, parity=1))
    num_unique = len(np.unique(x, axis=0))
    print(f"Odd parity,  group_size: {group_size}, num_unique: {num_unique}")

Even parity, group_size: 2, num_unique: 4096
Odd parity,  group_size: 2, num_unique: 4096
Even parity, group_size: 3, num_unique: 4096
Odd parity,  group_size: 3, num_unique: 4096
Even parity, group_size: 4, num_unique: 4096
Odd parity,  group_size: 4, num_unique: 4096
Even parity, group_size: 6, num_unique: 4096
Odd parity,  group_size: 6, num_unique: 4096
Even parity, group_size: 9, num_unique: 4096
Odd parity,  group_size: 9, num_unique: 4096
Even parity, group_size: 12, num_unique: 4096
Odd parity,  group_size: 12, num_unique: 4096
Even parity, group_size: 18, num_unique: 4096
Odd parity,  group_size: 18, num_unique: 4096
Even parity, group_size: 36, num_unique: 4096
Odd parity,  group_size: 36, num_unique: 4096


### Build and train diffusion model

In [ ]:
from core.DiT_model_lib import DiT
from core.diffusion_basics_lib import *
from core.diffusion_edm_lib import * 
from core.diffusion_esm_edm_lib import EDMDeltaGMMScoreLoss
import json
import math
from pprint import pprint
import pickle as pkl
from easydict import EasyDict as edict
from circuit_toolkit.plot_utils import to_imgrid

In [24]:
sample_num = 4096
sample_len = 36
group_size = 36
parity = 0
imgsize = int(math.sqrt(sample_len))
imgchannels = 1
parity_str = "even" if parity == 0 else "odd"
x = sample_ensuring_uniqueness(N=sample_num, sample_func=lambda N: sample_group_parity_vec(N=N, sample_len=sample_len, group_size=group_size, parity=parity))
print(x.shape)
dataset_name = f"parity_N{sample_num}_D{sample_len}_G{group_size}_{parity_str}"
print(dataset_name)
Xtsr = th.from_numpy(x).float()
Xtsr = Xtsr.view(sample_num, imgchannels, imgsize, imgsize,)

(4096, 36)
parity_N4096_D36_G36_even


In [29]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset_name=f"parity_N{sample_num}_D{sample_len}_G{group_size}_{parity_str}",
    patch_size = 1,
    hidden_size = 384,
    depth = 6,
    num_heads = 6,
    mlp_ratio = 4,
    class_dropout_prob = 0.1,
    num_classes = 0,
    learn_sigma = False,
    loss_type = "DSM",
)

In [31]:
eval_sample_size = 2048
imgshape = (1, 6, 6)
eval_batch_size = 1024
eval_fix_noise_seed = False
eval_sampling_steps = 35
lr = 1e-4
nsteps = 50000
batch_size = 256
record_frequency = 0
record_times = [*range(0, nsteps, 1000)]
save_ckpts = False
ckpt_step_list = []


saveroot = f"/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/DL_Projects/DiffusionParityLearning"
exp_name = f"DiT_mini_{args.dataset_name}"
savedir = f"{saveroot}/{exp_name}"
sample_dir = f"{savedir}/samples"
ckpt_dir = f"{savedir}/ckpts"
os.makedirs(savedir, exist_ok=True) 
os.makedirs(sample_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

In [32]:
loss_store = {}
def sampling_callback_fn(epoch, loss, model):
    loss_store[epoch] = loss
    x_out_batches = []
    if eval_fix_noise_seed:
        noise_init_all = torch.randn(eval_sample_size, *imgshape, generator=torch.Generator().manual_seed(0))
    else:
        noise_init_all = torch.randn(eval_sample_size, *imgshape)
    for i in range(0, eval_sample_size, eval_batch_size):
        batch_size_i = min(eval_batch_size, eval_sample_size - i)
        noise_init = noise_init_all[i:i+batch_size_i].to(device)
        x_out_i = edm_sampler(model, noise_init, num_steps=eval_sampling_steps, 
                        sigma_min=0.002, sigma_max=80, rho=7, return_traj=False)
        # x_out_i, x_traj_i, x0hat_traj_i, t_steps_i = edm_sampler(model, noise_init,
        #                 num_steps=eval_sampling_steps, sigma_min=0.002, sigma_max=80, rho=7, return_traj=True)
        x_out_batches.append(x_out_i)
    
    x_out = torch.cat(x_out_batches, dim=0)
    # sample_store[epoch] = x_out.cpu(), # x_traj.cpu(), x0hat_traj.cpu(), t_steps.cpu()
    torch.save(x_out, f"{sample_dir}/samples_epoch_{epoch:06d}.pt")
    mtg = to_imgrid(((x_out.cpu()[:64] + 1) / 2).clamp(0, 1), nrow=8, padding=1)
    mtg.save(f"{sample_dir}/samples_epoch_{epoch:06d}.png")

In [35]:
device = "cuda"
sigma_data = 1.0
# Xtsr_raw, imgsize, imgchannels = load_raw_dataset(dataset_name)
# Xtsr = (Xtsr_raw.to(device) - 0.5) / 0.5
pnts = Xtsr.view(Xtsr.shape[0], -1)
imgshape = Xtsr.shape[1:]
ndim = pnts.shape[1]
# cov_empirical = torch.cov(pnts.T, correction=1)
print(f"{args.dataset_name} dataset {Xtsr.shape[0]} samples, {ndim} features")
config = edict(
    input_size=imgsize,
    in_channels=imgchannels,
    patch_size=args.patch_size,
    hidden_size=args.hidden_size,
    depth=args.depth,
    num_heads=args.num_heads,
    mlp_ratio=args.mlp_ratio,
    class_dropout_prob=args.class_dropout_prob,
    num_classes=0,  # No class conditioning
    learn_sigma=False,
)
pprint(config)

json.dump(config, open(f"{savedir}/config.json", "w"))
json.dump(args.__dict__, open(f"{savedir}/args.json", "w"))

DiT_model = DiT(**config)
model_precd = EDMDiTPrecondWrapper(DiT_model, sigma_data=sigma_data, sigma_min=0.002, sigma_max=80, rho=7.0)
if args.loss_type == "DSM":
    edm_loss_fn = EDMLoss(P_mean=-1.2, P_std=1.2, sigma_data=sigma_data)
elif args.loss_type == "ESM":
    edm_loss_fn = EDMDeltaGMMScoreLoss(train_Xmat=Xtsr.to(device), P_mean=-1.2, P_std=1.2, sigma_data=sigma_data)
else:
    raise ValueError(f"Invalid loss type: {args.loss_type}")
model_precd, loss_traj = train_score_model_custom_loss(Xtsr, model_precd, edm_loss_fn, 
                                    lr=lr, nepochs=nsteps, batch_size=batch_size, device=device, 
                                    callback=sampling_callback_fn, callback_freq=record_frequency, callback_step_list=record_times,
                                    save_ckpts=save_ckpts, ckpt_dir=ckpt_dir, save_ckpt_step_list=ckpt_step_list)

pkl.dump(loss_store, open(f"{savedir}/loss_store.pkl", "wb"))
pkl.dump(loss_traj, open(f"{savedir}/loss_traj.pkl", "wb"))
torch.save(model_precd.model.state_dict(), f"{savedir}/model_final.pth")

noise_init = torch.randn(64, *imgshape).to(device)
x_out, x_traj, x0hat_traj, t_steps = edm_sampler(model_precd, noise_init, 
                num_steps=40, sigma_min=0.002, sigma_max=80, rho=7, return_traj=True)
mtg = to_imgrid(((x_out.cpu()[:]+1)/2).clamp(0, 1), nrow=8, padding=1)
mtg.save(f"{savedir}/learned_samples_final.png")

parity_N4096_D36_G36_even dataset 4096 samples, 36 features
{'class_dropout_prob': 0.1,
 'depth': 6,
 'hidden_size': 384,
 'in_channels': 1,
 'input_size': 6,
 'learn_sigma': False,
 'mlp_ratio': 4,
 'num_classes': 0,
 'num_heads': 6,
 'patch_size': 1}


  0%|          | 0/50000 [00:00<?, ?it/s]

step 0 loss 1.024
